In [1]:
pip install pillow speechrecognition pyttsx3


In [4]:
pip install requests pyautogui googletrans==4.0.0-rc1

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.3 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.3 MB ? eta -:--:--
   ---------------- ----------------------- 0.5/1.3 MB 644.1 kB/s eta 0:00:02
   -------------------------------- ------- 1.0/1.3 MB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 1.3 MB/s eta 0:00:00
  Created wheel for googletrans: filename=googletrans-4.0.0rc1-py3-none-any.whl size=17518 sha256=d70e18c0fc194c45002842872f69deee9fb2ed0adfa61f6508a9b8f071bc0c80
  Stored in directory: c:\users\sanjh\appdata\local\pip\cache\wheels\c0\59\9f\7372f0cf70160fe61b528532e1a7c8498c4becd6bcffb022de
Successfully built googletrans

   ------- --------------------------------  2/11 [hpack]
  Attempti

  DEPRECATION: Building 'googletrans' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'googletrans'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyterlab 4.4.3 requires httpx>=0.25.0, but you have httpx 0.13.3 which is incompatible.


In [2]:
pip install speechrecognition pyttsx3 pillow geocoder requests googletrans==4.0.0-rc1 pygame


   ------------- -------------------------- 1/3 [future]
   ------------- -------------------------- 1/3 [future]
   ------------- -------------------------- 1/3 [future]
   ------------- -------------------------- 1/3 [future]
   ------------- -------------------------- 1/3 [future]
   ------------- -------------------------- 1/3 [future]
   ------------- -------------------------- 1/3 [future]
   ------------- -------------------------- 1/3 [future]
   ------------- -------------------------- 1/3 [future]
   ------------- -------------------------- 1/3 [future]
   ------------- -------------------------- 1/3 [future]
   ------------- -------------------------- 1/3 [future]
   -------------------------- ------------- 2/3 [geocoder]
   -------------------------- ------------- 2/3 [geocoder]
   -------------------------- ------------- 2/3 [geocoder]
   ---------------------------------------- 3/3 [geocoder]

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
import tkinter as tk
import threading
import speech_recognition as sr
import pyttsx3
import time
import math
import datetime
import webbrowser
import random
import sys
import urllib.parse
import re
import os
import subprocess
import platform
from tkinter import ttk, font, messagebox
import geocoder
import requests
import pygame

# Translation service - handle googletrans compatibility issues
try:
    from googletrans import Translator, LANGUAGES
    TRANSLATION_AVAILABLE = True
except (ImportError, AttributeError) as e:
    print(f"Warning: googletrans not available ({e}). Translation features will be limited.")
    TRANSLATION_AVAILABLE = False
    LANGUAGES = {'english': 'en', 'spanish': 'es', 'french': 'fr', 'german': 'de', 
                'italian': 'it', 'portuguese': 'pt', 'russian': 'ru', 'japanese': 'ja',
                'chinese': 'zh', 'korean': 'ko', 'arabic': 'ar', 'hindi': 'hi'}
import logging
import json
import unittest
from math import sin, cos, tan, sqrt, pi

# Configure logging with enhanced format
logging.basicConfig(
    filename="aura.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode='a'
)

class SafeTTSEngine:
    """Thread-safe wrapper for pyttsx3 engine"""
    def __init__(self):
        self.engine = None
        self.lock = threading.Lock()
        self._initialize_engine()
    
    def _initialize_engine(self):
        try:
            self.engine = pyttsx3.init()
            self.engine.setProperty('rate', 170)
            self.engine.setProperty('volume', 0.95)
            
            # Set a clear voice if available
            voices = self.engine.getProperty('voices')
            if voices and len(voices) > 1:
                for voice in voices:
                    if any(keyword in voice.name.lower() for keyword in ["female", "zira", "hazel"]):
                        self.engine.setProperty('voice', voice.id)
                        break
                else:
                    self.engine.setProperty('voice', voices[1].id)
        except Exception as e:
            logging.error(f"TTS engine initialization failed: {e}")
            self.engine = None
    
    def speak(self, text):
        if not self.engine:
            logging.warning("TTS engine not available")
            return
            
        with self.lock:
            try:
                logging.info(f"Speaking: {text}")
                self.engine.say(text)
                self.engine.runAndWait()
            except Exception as e:
                logging.error(f"Speech error: {e}")
                try:
                    # Reinitialize engine on error
                    self._initialize_engine()
                    if self.engine:
                        self.engine.say(text)
                        self.engine.runAndWait()
                except Exception as retry_error:
                    logging.error(f"Speech retry failed: {retry_error}")

# Initialize TTS engine
tts_engine = SafeTTSEngine()

def speak(text):
    """Safe speak function"""
    tts_engine.speak(text)

# Input sanitization for file operations
def sanitize_filename(filename):
    """Remove invalid characters and prevent directory traversal."""
    if not filename:
        return ""
    # Remove dangerous characters and path traversal attempts
    sanitized = re.sub(r'[<>:"/\\|?*]', '', filename.strip())
    sanitized = sanitized.replace('..', '').replace('~', '')
    return sanitized[:255]  # Limit filename length

# Safe evaluation for math expressions
def safe_eval(expression):
    """Safely evaluate mathematical expressions"""
    allowed_names = {
        "sin": sin, "cos": cos, "tan": tan, "sqrt": sqrt, "pi": pi,
        "abs": abs, "round": round, "min": min, "max": max,
        "pow": pow, "log": math.log, "exp": math.exp
    }
    
    # Remove any non-mathematical characters
    safe_expr = re.sub(r'[^0-9+\-*/().sincostabrqpmlxe ]', '', expression.lower())
    
    try:
        return eval(safe_expr, {"__builtins__": {}}, allowed_names)
    except Exception as e:
        raise ValueError(f"Invalid expression: {str(e)}")

# Theme management
def save_theme(theme_name):
    """Save theme preference to a JSON file."""
    try:
        theme_data = {"theme": theme_name, "timestamp": time.time()}
        with open("theme_config.json", "w") as f:
            json.dump(theme_data, f)
        logging.info(f"Saved theme: {theme_name}")
    except Exception as e:
        logging.error(f"Error saving theme: {e}")

def load_theme():
    """Load theme preference from a JSON file."""
    try:
        with open("theme_config.json", "r") as f:
            data = json.load(f)
            return data.get("theme", "dark")
    except (FileNotFoundError, json.JSONDecodeError):
        return "dark"
    except Exception as e:
        logging.error(f"Error loading theme: {e}")
        return "dark"

# GUI Class with Modern Design
class AuraAssistant:
    def __init__(self, root):
        self.root = root
        self.root.title("AURA - Advanced Voice Assistant")
        self.root.geometry("1200x800")
        self.root.minsize(1000, 700)
        self.root.configure(bg='#0A0A0F')
        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)
        self.root.resizable(True, True)

        # Initialize variables before any GUI creation
        self.animate = True
        self.listening = True
        self.conversation_context = {}
        self.last_interaction_time = time.time()
        self.events = {}
        self.weather_api_key = os.getenv("WEATHER_API_KEY", "6af6d8086d6efb536328b65544bb91f8")
        
        # Animation variables
        self.angle = 0
        self.orb_radius = 25
        self.pulse_phase = 0
        self.stars = []
        self.particle_trails = []
        self.particles = []

        # Initialize services
        self._initialize_services()
        
        # Load theme first
        self.current_theme = load_theme()
        
        # Setup fonts
        self._setup_fonts()
        
        # Create enhanced GUI
        self._create_modern_gui()
        
        # Apply theme after GUI creation
        if self.current_theme == "light":
            self.toggle_theme()

        # Start services
        self._start_services()

    def _initialize_services(self):
        """Initialize external services"""
        try:
            if TRANSLATION_AVAILABLE:
                self.translator = Translator()
            else:
                self.translator = None
        except Exception as e:
            logging.error(f"Translator initialization failed: {e}")
            self.translator = None

        try:
            pygame.mixer.init(frequency=22050, size=-16, channels=2, buffer=512)
        except pygame.error as e:
            logging.error(f"Pygame mixer initialization failed: {e}")

    def _setup_fonts(self):
        """Setup modern typography"""
        try:
            self.title_font = font.Font(family="Segoe UI", size=32, weight="bold")
            self.subtitle_font = font.Font(family="Segoe UI", size=14, weight="normal")
            self.status_font = font.Font(family="Segoe UI", size=15, weight="normal")
            self.history_font = font.Font(family="Segoe UI", size=12, weight="normal")
            self.button_font = font.Font(family="Segoe UI", size=11, weight="bold")
            self.card_title_font = font.Font(family="Segoe UI", size=16, weight="bold")
            self.feature_font = font.Font(family="Segoe UI", size=12, weight="normal")
            self.small_font = font.Font(family="Segoe UI", size=10, weight="normal")
        except Exception as e:
            logging.error(f"Font setup error: {e}")
            # Fallback fonts
            self.title_font = ("Arial", 32, "bold")
            self.subtitle_font = ("Arial", 14)
            self.status_font = ("Arial", 15)
            self.history_font = ("Arial", 12)
            self.button_font = ("Arial", 11, "bold")
            self.card_title_font = ("Arial", 16, "bold")
            self.feature_font = ("Arial", 12)
            self.small_font = ("Arial", 10)

    def _create_modern_gui(self):
        """Create the redesigned modern GUI"""
        # Main container with gradient background
        self.main_container = tk.Frame(self.root, bg='#0A0A0F')
        self.main_container.pack(fill=tk.BOTH, expand=True, padx=20, pady=20)
        
        # Create header
        self._create_futuristic_header()
        
        # Create main content area with two columns
        content = tk.Frame(self.main_container, bg='#0A0A0F')
        content.pack(fill=tk.BOTH, expand=True, pady=20)
        
        # Left column - Visualization and Status
        left_column = tk.Frame(content, bg='#0A0A0F', width=450)
        left_column.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=(0, 10))
        left_column.pack_propagate(False)
        
        self._create_visualization_card(left_column)
        self._create_status_card(left_column)
        self._create_quick_actions_card(left_column)
        
        # Right column - Conversation and Tools
        right_column = tk.Frame(content, bg='#0A0A0F')
        right_column.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=(10, 0))
        
        self._create_conversation_card(right_column)
        self._create_translation_card(right_column)
        
        # Bottom bar
        self._create_bottom_bar()

    def _create_futuristic_header(self):
        """Create a stunning header with glass morphism effect"""
        header = tk.Frame(self.main_container, bg='#0A0A0F', height=100)
        header.pack(fill=tk.X)
        header.pack_propagate(False)
        
        # Glass morphism effect container
        glass = tk.Frame(header, bg='#14151F', highlightthickness=0)
        glass.place(relx=0, rely=0, relwidth=1, relheight=1)
        
        # Add subtle border
        border = tk.Frame(glass, bg='#2A2B3A', height=1)
        border.pack(side=tk.BOTTOM, fill=tk.X)
        
        # Left side - Logo and title
        left_section = tk.Frame(glass, bg='#14151F')
        left_section.pack(side=tk.LEFT, padx=30, pady=20)
        
        # Animated logo canvas
        self.logo_canvas = tk.Canvas(left_section, width=60, height=60, bg='#14151F', 
                                    highlightthickness=0)
        self.logo_canvas.pack(side=tk.LEFT, padx=(0, 20))
        
        # Create logo elements
        self._create_logo_animation()
        
        # Title section
        title_frame = tk.Frame(left_section, bg='#14151F')
        title_frame.pack(side=tk.LEFT)
        
        title = tk.Label(title_frame, text="AURA", font=self.title_font,
                        fg='#FFFFFF', bg='#14151F')
        title.pack(anchor='w')
        
        subtitle = tk.Label(title_frame, text="Advanced Voice Interface", 
                           font=self.subtitle_font, fg='#8A8FB9', bg='#14151F')
        subtitle.pack(anchor='w')
        
        # Right side - Controls
        right_section = tk.Frame(glass, bg='#14151F')
        right_section.pack(side=tk.RIGHT, padx=30, pady=20)
        
        # Status indicator
        status_frame = tk.Frame(right_section, bg='#14151F')
        status_frame.pack(side=tk.LEFT, padx=(0, 20))
        
        self.status_led = tk.Canvas(status_frame, width=12, height=12, bg='#14151F',
                                   highlightthickness=0)
        self.status_led.pack(side=tk.LEFT, padx=(0, 8))
        self.led = self.status_led.create_oval(2, 2, 10, 10, fill='#4ade80', outline='')
        
        self.status_label = tk.Label(status_frame, text="Active", font=self.small_font,
                                    fg='#4ade80', bg='#14151F')
        self.status_label.pack(side=tk.LEFT)
        
        # Theme toggle
        self.theme_btn = self._create_modern_button(right_section, "🌓 Theme", 
                                                    self.toggle_theme, '#6366F1', '#4F46E5')

    def _create_logo_animation(self):
        """Create animated logo elements"""
        # Outer ring
        self.logo_canvas.create_oval(5, 5, 55, 55, outline='#6366F1', width=2)
        
        # Inner glow
        for i in range(3):
            size = 45 - i*8
            offset = (60 - size) / 2
            self.logo_canvas.create_oval(offset, offset, offset+size, offset+size,
                                        outline=f'#{40+i*15:02x}{40+i*15:02x}F1', width=1)
        
        # Center dot
        self.logo_canvas.create_oval(27, 27, 33, 33, fill='#818CF8', outline='')

    def _create_visualization_card(self, parent):
        """Create stunning visualization card"""
        card = self._create_glass_card(parent, height=400)
        
        # Title
        title = tk.Label(card, text="Neural Core", font=self.card_title_font,
                        fg='#FFFFFF', bg='#14151F')
        title.pack(anchor='w', padx=20, pady=(15, 0))
        
        # Canvas container
        canvas_container = tk.Frame(card, bg='#14151F', height=320)
        canvas_container.pack(fill=tk.BOTH, expand=True, padx=20, pady=15)
        canvas_container.pack_propagate(False)
        
        # Main canvas
        self.canvas = tk.Canvas(canvas_container, width=380, height=300, 
                               bg='#0A0A0F', highlightthickness=0)
        self.canvas.pack(fill=tk.BOTH, expand=True)
        
        # Create visual elements
        self._create_visual_elements()

    def _create_visual_elements(self):
        """Create stunning visual elements"""
        center_x, center_y = 190, 150
        
        # Create starfield
        for _ in range(100):
            x = random.randint(10, 370)
            y = random.randint(10, 290)
            size = random.uniform(1, 3)
            brightness = random.randint(100, 255)
            color = f'#{brightness:02x}{brightness:02x}{brightness:02x}'
            self.stars.append({
                'id': self.canvas.create_oval(x, y, x+size, y+size, fill=color, outline=''),
                'x': x, 'y': y, 'size': size, 'brightness': brightness,
                'twinkle_speed': random.uniform(0.02, 0.05)
            })
        
        # Create central orb with multiple layers
        self.orb_layers = []
        colors = ['#4F46E5', '#6366F1', '#818CF8', '#A5B4FC']
        for i, color in enumerate(colors):
            radius = 40 - i*8
            obj = self.canvas.create_oval(center_x-radius, center_y-radius,
                                         center_x+radius, center_y+radius,
                                         fill=color, outline='', stipple='gray50' if i < 2 else '')
            self.orb_layers.append(obj)
        
        # Create orbital rings
        self.rings = []
        for i, radius in enumerate([60, 80, 100]):
            ring = self.canvas.create_oval(center_x-radius, center_y-radius,
                                          center_x+radius, center_y+radius,
                                          outline=f'#{80+i*30:02x}{80+i*30:02x}F1', 
                                          width=1, dash=(5, 3) if i > 0 else ())
            self.rings.append(ring)
        
        # Create particles
        self.particles = []
        for i in range(12):
            angle = i * (math.pi/6)
            distance = random.randint(70, 120)
            x = center_x + distance * math.cos(angle)
            y = center_y + distance * math.sin(angle)
            color = random.choice(['#F472B6', '#FBBF24', '#34D399', '#60A5FA'])
            particle = self.canvas.create_oval(x-3, y-3, x+3, y+3, fill=color, outline='')
            self.particles.append({
                'id': particle, 'angle': angle, 'distance': distance,
                'speed': random.uniform(0.01, 0.03), 'color': color
            })

    def _create_status_card(self, parent):
        """Create status display card"""
        card = self._create_glass_card(parent, height=80)
        
        status_frame = tk.Frame(card, bg='#14151F')
        status_frame.pack(fill=tk.BOTH, expand=True, padx=20, pady=15)
        
        # Status icon
        icon_label = tk.Label(status_frame, text="⚡", font=('Segoe UI', 20),
                             fg='#FBBF24', bg='#14151F')
        icon_label.pack(side=tk.LEFT, padx=(0, 15))
        
        # Status text
        self.status = tk.Label(status_frame, text="System ready. Listening...",
                              font=self.status_font, fg='#FFFFFF', bg='#14151F',
                              wraplength=300, justify=tk.LEFT)
        self.status.pack(side=tk.LEFT, fill=tk.X, expand=True)

    def _create_quick_actions_card(self, parent):
        """Create quick actions card"""
        card = self._create_glass_card(parent, height=200)
        
        title = tk.Label(card, text="Quick Actions", font=self.card_title_font,
                        fg='#FFFFFF', bg='#14151F')
        title.pack(anchor='w', padx=20, pady=(15, 0))
        
        # Action buttons grid
        grid = tk.Frame(card, bg='#14151F')
        grid.pack(fill=tk.BOTH, expand=True, padx=20, pady=15)
        
        actions = [
            ('🌤️ Weather', lambda: self.process_text_input("weather")),
            ('🧮 Calculate', lambda: self.process_text_input("calculate")),
            ('⏰ Timer', lambda: self.process_text_input("set timer for 5 minutes")),
            ('🎵 Music', lambda: self.process_text_input("play music on youtube")),
            ('🌐 Translate', self.show_translation_dialog),
            ('📅 Schedule', lambda: self.process_text_input("show calendar"))
        ]
        
        for i, (text, cmd) in enumerate(actions):
            row, col = i // 3, i % 3
            btn = tk.Button(grid, text=text, font=self.feature_font,
                          bg='#1E1F2B', fg='#FFFFFF', bd=0,
                          padx=15, pady=10, cursor='hand2',
                          activebackground='#2A2B3A', activeforeground='#FFFFFF',
                          command=cmd)
            btn.grid(row=row, column=col, padx=5, pady=5, sticky='nsew')
            
            # Hover effect
            btn.bind('<Enter>', lambda e, b=btn: b.configure(bg='#2A2B3A'))
            btn.bind('<Leave>', lambda e, b=btn: b.configure(bg='#1E1F2B'))
        
        # Configure grid weights
        for i in range(3):
            grid.columnconfigure(i, weight=1)
        for i in range(2):
            grid.rowconfigure(i, weight=1)

    def _create_conversation_card(self, parent):
        """Create conversation history card"""
        card = self._create_glass_card(parent, height=500)
        
        # Header with controls
        header = tk.Frame(card, bg='#14151F')
        header.pack(fill=tk.X, padx=20, pady=(15, 0))
        
        title = tk.Label(header, text="Conversation", font=self.card_title_font,
                        fg='#FFFFFF', bg='#14151F')
        title.pack(side=tk.LEFT)
        
        # Control buttons
        controls = tk.Frame(header, bg='#14151F')
        controls.pack(side=tk.RIGHT)
        
        clear_btn = tk.Button(controls, text='🗑️', font=('Segoe UI', 14),
                            bg='#EF4444', fg='#FFFFFF', bd=0,
                            padx=10, pady=2, cursor='hand2',
                            activebackground='#DC2626', activeforeground='#FFFFFF',
                            command=self.clear_history)
        clear_btn.pack(side=tk.LEFT, padx=2)
        
        export_btn = tk.Button(controls, text='💾', font=('Segoe UI', 14),
                             bg='#10B981', fg='#FFFFFF', bd=0,
                             padx=10, pady=2, cursor='hand2',
                             activebackground='#059669', activeforeground='#FFFFFF',
                             command=self.export_history)
        export_btn.pack(side=tk.LEFT, padx=2)
        
        # Conversation text area
        text_container = tk.Frame(card, bg='#0A0A0F')
        text_container.pack(fill=tk.BOTH, expand=True, padx=20, pady=15)
        
        self.history = tk.Text(text_container, bg='#0A0A0F', fg='#FFFFFF',
                              font=self.history_font, wrap=tk.WORD,
                              bd=0, padx=15, pady=15,
                              selectbackground='#4F46E5',
                              insertbackground='#FFFFFF')
        
        # Scrollbar
        scrollbar = tk.Scrollbar(text_container, bg='#2A2B3A',
                                troughcolor='#0A0A0F',
                                activebackground='#4F46E5',
                                command=self.history.yview)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        
        self.history.config(yscrollcommand=scrollbar.set)
        self.history.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        self.history.config(state=tk.DISABLED)
        
        # Text tags
        self.history.tag_configure("aura", foreground='#34D399', font=('Segoe UI', 12, 'normal'))
        self.history.tag_configure("user", foreground='#60A5FA', font=('Segoe UI', 12, 'normal'))
        self.history.tag_configure("timestamp", foreground='#6B7280', font=('Segoe UI', 10, 'normal'))

    def _create_translation_card(self, parent):
        """Create translation card"""
        card = self._create_glass_card(parent, height=250)
        
        title = tk.Label(card, text="Universal Translator", font=self.card_title_font,
                        fg='#FFFFFF', bg='#14151F')
        title.pack(anchor='w', padx=20, pady=(15, 0))
        
        content = tk.Frame(card, bg='#14151F')
        content.pack(fill=tk.BOTH, expand=True, padx=20, pady=15)
        
        # Source text
        self.translation_input = tk.Text(content, height=3,
                                        bg='#1E1F2B', fg='#FFFFFF',
                                        font=self.history_font,
                                        wrap=tk.WORD, bd=0,
                                        padx=10, pady=8,
                                        insertbackground='#FFFFFF')
        self.translation_input.pack(fill=tk.X, pady=(0, 10))
        
        # Language selection
        lang_frame = tk.Frame(content, bg='#14151F')
        lang_frame.pack(fill=tk.X, pady=(0, 10))
        
        tk.Label(lang_frame, text="To:", font=self.small_font,
                fg='#8A8FB9', bg='#14151F').pack(side=tk.LEFT, padx=(0, 10))
        
        self.target_language = ttk.Combobox(lang_frame, font=self.small_font,
                                           state="readonly", width=15)
        self.target_language.pack(side=tk.LEFT)
        
        # Populate languages
        if TRANSLATION_AVAILABLE:
            popular_languages = ['spanish', 'french', 'german', 'italian', 'portuguese', 
                                'russian', 'japanese', 'chinese', 'korean', 'arabic']
            self.target_language['values'] = popular_languages
            self.target_language.set('spanish')
        else:
            self.target_language['values'] = list(LANGUAGES.keys())
            self.target_language.set('spanish')
        
        # Translate button
        translate_btn = tk.Button(content, text="Translate →", font=self.button_font,
                                 bg='#6366F1', fg='#FFFFFF', bd=0,
                                 padx=20, pady=8, cursor='hand2',
                                 activebackground='#4F46E5', activeforeground='#FFFFFF',
                                 command=self.translate_text)
        translate_btn.pack(pady=(5, 10))
        
        # Result
        self.translation_result = tk.Text(content, height=2,
                                         bg='#1E1F2B', fg='#34D399',
                                         font=self.history_font,
                                         wrap=tk.WORD, bd=0,
                                         padx=10, pady=8,
                                         state=tk.DISABLED)
        self.translation_result.pack(fill=tk.X)

    def _create_bottom_bar(self):
        """Create bottom input bar"""
        bar = tk.Frame(self.main_container, bg='#14151F', height=80)
        bar.pack(fill=tk.X, pady=(10, 0))
        bar.pack_propagate(False)
        
        # Add subtle top border
        border = tk.Frame(bar, bg='#2A2B3A', height=1)
        border.pack(fill=tk.X)
        
        # Content
        content = tk.Frame(bar, bg='#14151F')
        content.pack(fill=tk.BOTH, expand=True, padx=20, pady=15)
        
        # Input field with modern styling
        input_container = tk.Frame(content, bg='#1E1F2B')
        input_container.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=(0, 15))
        
        self.text_input = tk.Entry(input_container, bg='#1E1F2B', fg='#FFFFFF',
                                  font=self.status_font, bd=0,
                                  insertbackground='#FFFFFF')
        self.text_input.pack(fill=tk.BOTH, expand=True, padx=15, pady=10)
        self.text_input.bind("<Return>", lambda e: self.process_text_command())
        
        # Send button
        send_btn = tk.Button(content, text="Send", font=self.button_font,
                            bg='#6366F1', fg='#FFFFFF', bd=0,
                            padx=30, pady=8, cursor='hand2',
                            activebackground='#4F46E5', activeforeground='#FFFFFF',
                            command=self.process_text_command)
        send_btn.pack(side=tk.RIGHT)
        
        # Voice indicator
        voice_indicator = tk.Label(content, text="🎤", font=('Segoe UI', 20),
                                  fg='#34D399', bg='#14151F')
        voice_indicator.pack(side=tk.RIGHT, padx=(0, 15))

    def _create_glass_card(self, parent, height=None):
        """Create a glass morphism card"""
        card = tk.Frame(parent, bg='#14151F', highlightthickness=0)
        if height:
            card.config(height=height)
        card.pack(fill=tk.X, pady=(0, 15))
        card.pack_propagate(False)
        
        # Add subtle border
        border = tk.Frame(card, bg='#2A2B3A', height=1)
        border.pack(side=tk.BOTTOM, fill=tk.X)
        
        return card

    def _create_modern_button(self, parent, text, command, bg, hover_bg):
        """Create a modern styled button"""
        btn = tk.Button(parent, text=text, font=self.button_font,
                       bg=bg, fg='#FFFFFF', bd=0,
                       padx=20, pady=8, cursor='hand2',
                       activebackground=hover_bg, activeforeground='#FFFFFF',
                       command=command)
        btn.pack(side=tk.LEFT, padx=5)
        
        # Hover effect
        btn.bind('<Enter>', lambda e: btn.configure(bg=hover_bg))
        btn.bind('<Leave>', lambda e: btn.configure(bg=bg))
        
        return btn

    def animate_orb(self):
        """Enhanced orb animation"""
        if not self.animate:
            return
        
        try:
            self.angle += 0.02
            self.pulse_phase += 0.03
            
            center_x, center_y = 190, 150
            
            # Pulse effect
            pulse = 1 + 0.1 * math.sin(self.pulse_phase)
            
            # Update orb layers with pulse
            for i, layer in enumerate(self.orb_layers):
                size = (40 - i*8) * pulse
                self.canvas.coords(layer,
                                  center_x-size, center_y-size,
                                  center_x+size, center_y+size)
            
            # Rotate rings
            for i, ring in enumerate(self.rings):
                rotation = self.angle * (i + 1) * 0.5
                radius = 60 + i*20 + 10 * math.sin(self.angle * 0.5)
                
                self.canvas.coords(ring,
                                  center_x-radius, center_y-radius,
                                  center_x+radius, center_y+radius)
            
            # Animate particles
            for particle in self.particles:
                particle['angle'] += particle['speed']
                x = center_x + particle['distance'] * math.cos(particle['angle'])
                y = center_y + particle['distance'] * math.sin(particle['angle'])
                
                self.canvas.coords(particle['id'],
                                  x-3, y-3, x+3, y+3)
            
            # Twinkle stars
            for star in self.stars:
                twinkle = 0.7 + 0.3 * math.sin(self.pulse_phase * star['twinkle_speed'] * 100)
                brightness = int(star['brightness'] * twinkle)
                color = f'#{brightness:02x}{brightness:02x}{brightness:02x}'
                self.canvas.itemconfig(star['id'], fill=color)
            
            # Update listening indicator
            if self.listening:
                self.status_led.itemconfig(self.led, fill='#4ade80')
                self.status_label.config(text='Active', fg='#4ade80')
            else:
                self.status_led.itemconfig(self.led, fill='#fbbf24')
                self.status_label.config(text='Processing', fg='#fbbf24')
        
        except Exception as e:
            logging.error(f"Animation error: {e}")
        
        if self.animate:
            self.root.after(16, self.animate_orb)

    def toggle_theme(self):
        """Toggle between themes"""
        if self.current_theme == "dark":
            self._apply_light_theme()
            self.current_theme = "light"
            self.theme_btn.config(text="🌑 Dark")
        else:
            self._apply_dark_theme()
            self.current_theme = "dark"
            self.theme_btn.config(text="🌓 Theme")
        
        save_theme(self.current_theme)

    def _apply_dark_theme(self):
        """Apply dark theme"""
        self.root.configure(bg='#0A0A0F')
        self._update_theme_colors('#0A0A0F', '#14151F', '#1E1F2B', '#FFFFFF')

    def _apply_light_theme(self):
        """Apply light theme"""
        self.root.configure(bg='#F3F4F6')
        self._update_theme_colors('#F3F4F6', '#FFFFFF', '#F9FAFB', '#1F2937')

    def _update_theme_colors(self, bg, card_bg, input_bg, fg):
        """Update theme colors throughout the app"""
        try:
            # Update main container
            self.main_container.configure(bg=bg)
            
            # Update all widgets (simplified - in production you'd traverse all widgets)
            for widget in self.main_container.winfo_children():
                self._recursive_theme_update(widget, bg, card_bg, input_bg, fg)
        except Exception as e:
            logging.error(f"Theme update error: {e}")

    def _recursive_theme_update(self, widget, bg, card_bg, input_bg, fg):
        """Recursively update widget colors"""
        try:
            if isinstance(widget, (tk.Frame, tk.LabelFrame)):
                if widget.cget('bg') in ['#0A0A0F', '#14151F', '#1E1F2B', '#F3F4F6', '#FFFFFF', '#F9FAFB']:
                    if 'card' in str(widget) or widget.cget('bg') in ['#14151F', '#FFFFFF']:
                        widget.configure(bg=card_bg)
                    elif widget.cget('bg') in ['#1E1F2B', '#F9FAFB']:
                        widget.configure(bg=input_bg)
                    else:
                        widget.configure(bg=bg)
            
            elif isinstance(widget, tk.Label):
                if widget.cget('bg') in ['#0A0A0F', '#14151F', '#1E1F2B', '#F3F4F6', '#FFFFFF', '#F9FAFB']:
                    widget.configure(bg=card_bg if 'card' in str(widget) else bg)
                if widget.cget('fg') in ['#FFFFFF', '#1F2937', '#8A8FB9']:
                    widget.configure(fg=fg)
            
            elif isinstance(widget, tk.Text) and widget != self.history:
                widget.configure(bg=input_bg, fg=fg)
            
            # Recurse
            for child in widget.winfo_children():
                self._recursive_theme_update(child, bg, card_bg, input_bg, fg)
        except:
            pass

    def show_translation_dialog(self):
        """Show translation dialog"""
        self.translation_input.focus()

    def translate_text(self):
        """Perform translation"""
        try:
            text = self.translation_input.get("1.0", tk.END).strip()
            if not text:
                self.update_translation_result("Please enter text to translate.")
                return
            
            target_lang = self.target_language.get()
            if not target_lang:
                self.update_translation_result("Please select a target language.")
                return
            
            if not TRANSLATION_AVAILABLE or not self.translator:
                self.update_translation_result("Translation service not available.")
                return
            
            lang_code = LANGUAGES.get(target_lang, target_lang)
            translation = self.translator.translate(text, dest=lang_code)
            result = f"'{text}' → '{translation.text}'"
            
            self.update_translation_result(result)
            self.add_to_history(f"Translation: {result}", "aura")
            threading.Thread(target=speak, args=(f"Translation: {translation.text}",), daemon=True).start()
            
        except Exception as e:
            logging.error(f"Translation GUI error: {e}")
            self.update_translation_result("Translation failed. Please try again.")

    def update_translation_result(self, text):
        """Update translation result display"""
        try:
            self.translation_result.config(state=tk.NORMAL)
            self.translation_result.delete("1.0", tk.END)
            self.translation_result.insert("1.0", text)
            self.translation_result.config(state=tk.DISABLED)
        except Exception as e:
            logging.error(f"Translation result update error: {e}")

    def export_history(self):
        """Export conversation history"""
        try:
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"conversation_export_{timestamp}.txt"
            
            content = self.history.get("1.0", tk.END)
            with open(filename, "w", encoding="utf-8") as f:
                f.write(f"AURA Conversation Export\n")
                f.write(f"Generated: {datetime.datetime.now()}\n")
                f.write("=" * 50 + "\n\n")
                f.write(content)
            
            self.set_status(f"History exported to {filename}")
            self.add_to_history(f"Aura: History exported to {filename}", "aura")
            
        except Exception as e:
            logging.error(f"Export error: {e}")
            self.set_status("Failed to export history.")

    def get_greeting(self):
        """Get time-appropriate greeting"""
        hour = datetime.datetime.now().hour
        if 5 <= hour < 12:
            return "Good morning! I'm ready to assist you."
        elif 12 <= hour < 17:
            return "Good afternoon! How can I help you?"
        elif 17 <= hour < 21:
            return "Good evening! What can I do for you?"
        else:
            return "Hello! I'm here to help, whatever you need."

    def play_sound(self):
        """Play startup sound if available"""
        try:
            if os.path.exists("startup.wav"):
                sound = pygame.mixer.Sound("startup.wav")
                sound.play()
        except Exception as e:
            logging.error(f"Sound play error: {e}")

    def clear_history(self):
        """Clear conversation history"""
        try:
            self.history.config(state=tk.NORMAL)
            self.history.delete(1.0, tk.END)
            self.history.config(state=tk.DISABLED)
            self.add_to_history("Aura: Conversation history cleared.", "aura")
        except Exception as e:
            logging.error(f"History clear error: {e}")

    def set_timer(self, duration_str):
        """Set a timer"""
        try:
            duration = int(re.search(r'\d+', duration_str).group())
            if duration <= 0 or duration > 480:
                return "Timer duration must be between 1 and 480 minutes."
            
            def timer_callback():
                message = f"⏰ Timer for {duration} minutes has finished!"
                speak(message)
                self.set_status(message)
                self.add_to_history(f"Aura: {message}", "aura")
            
            timer = threading.Timer(duration * 60, timer_callback)
            timer.daemon = True
            timer.start()
            
            return f"✅ Timer set for {duration} minutes."
            
        except (ValueError, AttributeError):
            return "Please specify duration in minutes (e.g., 'set timer for 5 minutes')."
        except Exception as e:
            logging.error(f"Timer error: {e}")
            return "Error setting timer."

    def check_reminders(self):
        """Check for scheduled reminders"""
        while self.animate:
            try:
                now = datetime.datetime.now()
                expired_events = []
                
                for name, event_time in self.events.items():
                    if now >= event_time:
                        message = f"📅 Reminder: {name}"
                        speak(message)
                        self.set_status(message)
                        self.add_to_history(f"Aura: {message}", "aura")
                        expired_events.append(name)
                
                for name in expired_events:
                    del self.events[name]
                    
                time.sleep(30)
            except Exception as e:
                logging.error(f"Reminder check error: {e}")
                time.sleep(60)

    def process_text_input(self, command):
        """Process text command programmatically"""
        self.add_to_history(f"You: {command}", "user")
        response = self.process_command(command)
        self.set_status(response)
        self.add_to_history(f"Aura: {response}", "aura")
        threading.Thread(target=speak, args=(response,), daemon=True).start()

    def process_text_command(self):
        """Process text command from input"""
        try:
            command = self.text_input.get().strip()
            if not command:
                return
                
            self.text_input.delete(0, tk.END)
            self.add_to_history(f"You: {command}", "user")
            
            response = self.process_command(command.lower())
            self.set_status(response)
            self.add_to_history(f"Aura: {response}", "aura")
            
            threading.Thread(target=speak, args=(response,), daemon=True).start()
        except Exception as e:
            logging.error(f"Text command processing error: {e}")
            self.set_status("Error processing command.")

    def voice_listener(self):
        """Voice recognition loop"""
        recognizer = sr.Recognizer()
        recognizer.pause_threshold = 1.0
        recognizer.dynamic_energy_threshold = True
        
        try:
            microphone = sr.Microphone()
        except Exception as e:
            logging.error(f"Microphone initialization failed: {e}")
            self.set_status("🎤 Microphone not available. Using text input only.")
            return

        while self.animate:
            try:
                with microphone as source:
                    recognizer.adjust_for_ambient_noise(source, duration=1)
                    self.set_status("🎧 Listening...")
                    
                    audio = recognizer.listen(source, timeout=10, phrase_time_limit=15)
                    
                    try:
                        command = recognizer.recognize_google(audio, language='en-US').lower()
                        self.add_to_history(f"You: {command}", "user")
                        logging.info(f"Voice command: {command}")
                        
                        self.last_interaction_time = time.time()
                        response = self.process_command(command)
                        
                        self.set_status(response)
                        self.add_to_history(f"Aura: {response}", "aura")
                        threading.Thread(target=speak, args=(response,), daemon=True).start()
                        
                    except sr.UnknownValueError:
                        self.set_status("🤔 Sorry, I couldn't understand that. Please try again.")
                        time.sleep(1)
                    except sr.RequestError as e:
                        logging.error(f"Speech recognition service error: {e}")
                        self.set_status("🚫 Speech service temporarily unavailable. Please use text input.")
                        time.sleep(5)
                        
            except sr.WaitTimeoutError:
                if time.time() - self.last_interaction_time > 300:
                    self.set_status("💤 Still here when you need me.")
                    self.last_interaction_time = time.time()
                else:
                    self.set_status("🎧 Listening...")
                    
            except Exception as e:
                logging.error(f"Voice listener error: {e}")
                time.sleep(2)

    def get_user_location(self):
        """Get user's location"""
        try:
            g = geocoder.ip('me')
            return g.city or "unknown", g.latlng or [0, 0]
        except Exception as e:
            logging.error(f"Location detection failed: {e}")
            return "unknown", [0, 0]

    def process_command(self, command):
        """Process voice or text commands"""
        logging.info(f"Processing: {command}")
        
        try:
            if any(word in command for word in ['calculate', 'compute', 'math']):
                return self.handle_calculation(command)
            if 'timer' in command and any(word in command for word in ['set', 'start']):
                return self.handle_timer(command)
            if 'weather' in command:
                return self.handle_weather(command)
            if 'translate' in command:
                return self.handle_translation(command)
            if any(word in command for word in ['open', 'go to', 'visit']):
                return self.handle_web_commands(command)
            if 'search' in command or 'google' in command:
                return self.handle_search(command)
            if 'file' in command:
                return self.handle_file_operations(command)
            if any(word in command for word in ['schedule', 'remind', 'event', 'calendar']):
                return self.handle_calendar(command)
            if 'music' in command or 'play' in command:
                return self.handle_music(command)
            if any(word in command for word in ['time', 'date', 'joke']):
                return self.handle_system_commands(command)
            if any(word in command for word in ['navigate', 'directions', 'route']):
                return self.handle_navigation(command)
            return self.handle_conversation(command)
            
        except Exception as e:
            logging.error(f"Command processing error: {e}")
            return "⚠️ I encountered an error processing that command. Please try again."

    def handle_calculation(self, command):
        """Handle mathematical calculations"""
        try:
            expression = re.sub(r'(calculate|compute|math|what is|equals)', '', command).strip()
            
            replacements = {
                'plus': '+', 'add': '+', 'and': '+',
                'minus': '-', 'subtract': '-', 'take away': '-',
                'times': '*', 'multiply': '*', 'multiplied by': '*',
                'divide': '/', 'divided by': '/', 'over': '/',
                'power': '**', 'to the power of': '**',
                'squared': '**2', 'cubed': '**3'
            }
            
            for word, symbol in replacements.items():
                expression = expression.replace(word, symbol)
            
            result = safe_eval(expression)
            return f"🧮 The result is {result}."
            
        except Exception as e:
            return f"❌ I couldn't calculate that. Please check your expression."

    def handle_timer(self, command):
        """Handle timer commands"""
        try:
            numbers = re.findall(r'\d+', command)
            if not numbers:
                return "Please specify the timer duration in minutes."
            
            duration = int(numbers[0])
            return self.set_timer(str(duration))
            
        except Exception as e:
            return "❌ Error setting timer. Please specify duration in minutes."

    def handle_weather(self, command):
        """Handle weather requests"""
        try:
            city_match = re.search(r'weather\s+(?:in\s+|for\s+)?(.+)', command)
            if city_match:
                city_name = city_match.group(1).strip()
            else:
                city_name, _ = self.get_user_location()
                if city_name == "unknown":
                    return "Please specify a city for weather information."
            
            url = f"http://api.openweathermap.org/data/2.5/weather"
            params = {
                'q': city_name,
                'appid': self.weather_api_key,
                'units': 'metric'
            }
            
            response = requests.get(url, params=params, timeout=10)
            data = response.json()
            
            if response.status_code != 200:
                return f"🌍 Weather data not available for {city_name}. Please check the city name."
            
            weather_desc = data['weather'][0]['description'].title()
            temp = round(data['main']['temp'])
            feels_like = round(data['main']['feels_like'])
            humidity = data['main']['humidity']
            
            weather_emojis = {
                'clear': '☀️', 'sunny': '☀️', 'clouds': '☁️', 'cloudy': '☁️',
                'rain': '🌧️', 'drizzle': '🌦️', 'thunderstorm': '⛈️',
                'snow': '❄️', 'mist': '🌫️', 'fog': '🌫️'
            }
            
            emoji = '🌤️'
            for key, em in weather_emojis.items():
                if key in weather_desc.lower():
                    emoji = em
                    break
            
            return f"{emoji} Weather in {city_name}: {weather_desc}, {temp}°C (feels like {feels_like}°C), humidity {humidity}%."
            
        except requests.RequestException as e:
            logging.error(f"Weather API error: {e}")
            return "🌐 Weather service temporarily unavailable."
        except Exception as e:
            logging.error(f"Weather handling error: {e}")
            return "❌ Error retrieving weather information."

    def handle_translation(self, command):
        """Handle translation requests"""
        if not TRANSLATION_AVAILABLE or not self.translator:
            return "🌐 Translation service not available. Please install googletrans: pip install googletrans==4.0.0-rc1"
        
        try:
            match = re.search(r'translate\s+(.+?)\s+to\s+(\w+)', command)
            if not match:
                return "🔄 Please use format: 'translate [text] to [language]' or use the translation panel"
            
            text, target_lang = match.groups()
            
            lang_map = {name.lower(): code for name, code in LANGUAGES.items()}
            target_code = lang_map.get(target_lang.lower())
            
            if not target_code:
                available_langs = ', '.join(sorted(list(LANGUAGES.keys())[:10]))
                return f"🌍 Language '{target_lang}' not supported. Try: {available_langs}"
            
            translation = self.translator.translate(text, dest=target_code)
            result = f"🔄 '{text}' in {target_lang} is: '{translation.text}'"
            
            self.update_translation_result(f"'{text}' → '{translation.text}'")
            
            return result
            
        except Exception as e:
            logging.error(f"Translation error: {e}")
            return "❌ Translation failed. Please try again later."

    def handle_web_commands(self, command):
        """Handle web browsing commands"""
        try:
            sites = {
                'youtube': 'https://www.youtube.com',
                'google': 'https://www.google.com',
                'gmail': 'https://mail.google.com',
                'maps': 'https://www.google.com/maps',
                'news': 'https://news.google.com',
                'drive': 'https://drive.google.com',
                'docs': 'https://docs.google.com',
                'chatgpt': 'https://chat.openai.com',
                'wikipedia': 'https://www.wikipedia.org',
                'reddit': 'https://www.reddit.com',
                'twitter': 'https://www.twitter.com',
                'facebook': 'https://www.facebook.com',
                'instagram': 'https://www.instagram.com',
                'whatsapp': 'https://web.whatsapp.com',
                'spotify': 'https://open.spotify.com',
                'netflix': 'https://www.netflix.com'
            }
            
            for site, url in sites.items():
                if site in command:
                    webbrowser.open(url)
                    return f"🌐 Opening {site.title()}."
            
            url_pattern = r'(https?://[^\s]+|www\.[^\s]+|[^\s]+\.[a-z]{2,})'
            url_match = re.search(url_pattern, command)
            if url_match:
                url = url_match.group(1)
                if not url.startswith('http'):
                    url = 'https://' + url
                webbrowser.open(url)
                return f"🌐 Opening {url}."
            
            return "🌐 Please specify a website to open (e.g., 'open YouTube' or 'open google.com')."
            
        except Exception as e:
            logging.error(f"Web command error: {e}")
            return "❌ Error opening website."

    def handle_search(self, command):
        """Handle search commands"""
        try:
            query_patterns = [
                r'search\s+(?:for\s+)?(.+)',
                r'google\s+(.+)',
                r'look\s+up\s+(.+)'
            ]
            
            query = None
            for pattern in query_patterns:
                match = re.search(pattern, command)
                if match:
                    query = match.group(1).strip()
                    break
            
            if not query:
                return "🔍 Please specify what to search for."
            
            search_url = f"https://www.google.com/search?q={urllib.parse.quote(query)}"
            webbrowser.open(search_url)
            return f"🔍 Searching for '{query}' on Google."
            
        except Exception as e:
            logging.error(f"Search error: {e}")
            return "❌ Error performing search."

    def handle_file_operations(self, command):
        """Handle file operations safely"""
        try:
            if 'open' in command:
                filename = command.replace('open file', '').replace('open', '').strip()
                filename = sanitize_filename(filename)
                
                if not filename:
                    return "📁 Please specify a filename."
                
                if os.path.exists(filename):
                    if platform.system() == "Windows":
                        os.startfile(filename)
                    elif platform.system() == "Darwin":
                        subprocess.run(["open", filename])
                    else:
                        subprocess.run(["xdg-open", filename])
                    return f"📂 Opening file: {filename}"
                else:
                    return f"❌ File '{filename}' not found."
            
            elif 'list' in command:
                try:
                    files = [f for f in os.listdir('.') if os.path.isfile(f)][:10]
                    if files:
                        return f"📋 Recent files: {', '.join(files)}"
                    else:
                        return "📁 No files found in current directory."
                except PermissionError:
                    return "🚫 Permission denied to list files."
            
            elif 'create' in command:
                filename = command.replace('create file', '').replace('create', '').strip()
                filename = sanitize_filename(filename)
                
                if not filename:
                    return "📝 Please specify a filename."
                
                try:
                    with open(filename, 'w') as f:
                        f.write("")
                    return f"✅ Created file: {filename}"
                except PermissionError:
                    return "🚫 Permission denied to create file."
            
            return "📁 File operation not recognized. Try 'open file', 'list files', or 'create file'."
            
        except Exception as e:
            logging.error(f"File operation error: {e}")
            return "❌ Error with file operation."

    def handle_calendar(self, command):
        """Handle calendar and scheduling"""
        try:
            if 'schedule' in command or 'remind' in command:
                event_match = re.search(r'(?:schedule|remind).+?(.+?)\s+(?:at|on)\s+(.+)', command)
                if event_match:
                    event_name, time_str = event_match.groups()
                    try:
                        formats = ['%Y-%m-%d %H:%M', '%m/%d/%Y %H:%M', '%B %d %H:%M']
                        event_time = None
                        
                        for fmt in formats:
                            try:
                                event_time = datetime.datetime.strptime(time_str.strip(), fmt)
                                break
                            except ValueError:
                                continue
                        
                        if not event_time:
                            return "📅 Please use format: YYYY-MM-DD HH:MM or MM/DD/YYYY HH:MM"
                        
                        self.events[event_name.strip()] = event_time
                        return f"✅ Scheduled '{event_name}' for {event_time.strftime('%A, %B %d at %I:%M %p')}"
                        
                    except Exception:
                        return "❌ Invalid date format. Please use: YYYY-MM-DD HH:MM"
                else:
                    return "📅 Please specify event name and time (e.g., 'schedule meeting at 2024-12-01 14:30')"
            
            elif 'show' in command and 'calendar' in command:
                if not self.events:
                    return "📅 No events scheduled."
                
                event_list = []
                for name, time in sorted(self.events.items(), key=lambda x: x[1]):
                    event_list.append(f"• {name}: {time.strftime('%B %d at %I:%M %p')}")
                
                return "📅 Scheduled events:\n" + "\n".join(event_list)
            
            elif 'today' in command:
                today = datetime.datetime.now().date()
                today_events = {name: time for name, time in self.events.items() 
                              if time.date() == today}
                
                if not today_events:
                    return "📅 No events scheduled for today."
                
                event_list = []
                for name, time in sorted(today_events.items(), key=lambda x: x[1]):
                    event_list.append(f"• {name}: {time.strftime('%I:%M %p')}")
                
                return "📅 Today's events:\n" + "\n".join(event_list)
            
            return "📅 Calendar command not recognized. Try 'schedule [event] at [time]', 'show calendar', or 'today's events'."
            
        except Exception as e:
            logging.error(f"Calendar error: {e}")
            return "❌ Error with calendar operation."

    def handle_music(self, command):
        """Handle music commands"""
        try:
            if 'youtube' in command:
                song_match = re.search(r'play\s+(.+?)\s+(?:on\s+)?youtube', command)
                if song_match:
                    song = song_match.group(1).strip()
                    search_url = f"https://www.youtube.com/results?search_query={urllib.parse.quote(song)}"
                    webbrowser.open(search_url)
                    return f"🎵 Searching for '{song}' on YouTube."
                else:
                    webbrowser.open("https://www.youtube.com")
                    return "🎵 Opening YouTube."
            
            elif 'stop' in command:
                try:
                    pygame.mixer.music.stop()
                    return "⏹️ Music stopped."
                except:
                    return "❌ No music is currently playing."
            
            elif 'pause' in command:
                try:
                    pygame.mixer.music.pause()
                    return "⏸️ Music paused."
                except:
                    return "❌ No music is currently playing."
            
            elif 'resume' in command:
                try:
                    pygame.mixer.music.unpause()
                    return "▶️ Music resumed."
                except:
                    return "❌ No music to resume."
            
            elif 'play' in command:
                file_match = re.search(r'play\s+(.+)', command)
                if file_match:
                    filename = sanitize_filename(file_match.group(1).strip())
                    if os.path.exists(filename):
                        try:
                            pygame.mixer.music.load(filename)
                            pygame.mixer.music.play()
                            return f"🎵 Playing: {filename}"
                        except pygame.error:
                            return f"❌ Cannot play file: {filename}"
                    else:
                        return f"❌ Music file '{filename}' not found."
                
                return "🎵 Please specify a music file to play."
            
            return "🎵 Music command not recognized. Try 'play [song] on YouTube' or 'stop music'."
            
        except Exception as e:
            logging.error(f"Music error: {e}")
            return "❌ Error with music operation."

    def handle_system_commands(self, command):
        """Handle system information commands"""
        try:
            if 'time' in command:
                current_time = datetime.datetime.now().strftime("%I:%M %p")
                return f"⏰ The current time is {current_time}."
            
            elif 'date' in command:
                current_date = datetime.datetime.now().strftime("%A, %B %d, %Y")
                return f"📅 Today is {current_date}."
            
            elif 'joke' in command:
                jokes = [
                    "Why did the AI go to therapy? It had deep learning issues!",
                    "Why don't programmers like nature? It has too many bugs.",
                    "What do you call a computer that sings? A Dell!",
                    "Why did the computer keep sneezing? It had a virus!",
                    "How do you comfort a JavaScript bug? You console it!",
                    "Why do Python programmers prefer dark mode? Because light attracts bugs!",
                    "What's a computer's favorite beat? An algo-rhythm!",
                    "Why did the computer go to art school? To improve its graphics!"
                ]
                return f"😄 {random.choice(jokes)}"
            
            return "❓ System command not recognized."
            
        except Exception as e:
            logging.error(f"System command error: {e}")
            return "❌ Error processing system command."

    def handle_navigation(self, command):
        """Handle navigation commands"""
        try:
            dest_match = re.search(r'(?:navigate to|directions to|route to)\s+(.+)', command)
            if not dest_match:
                return "🗺️ Please specify a destination."
            
            destination = dest_match.group(1).strip()
            city, coords = self.get_user_location()
            
            if city == "unknown":
                maps_url = f"https://www.google.com/maps/search/{urllib.parse.quote(destination)}"
            else:
                maps_url = f"https://www.google.com/maps/dir/{coords[0]},{coords[1]}/{urllib.parse.quote(destination)}"
            
            webbrowser.open(maps_url)
            return f"🗺️ Opening directions to {destination}."
            
        except Exception as e:
            logging.error(f"Navigation error: {e}")
            return "❌ Error with navigation request."

    def handle_conversation(self, command):
        """Handle conversational commands"""
        try:
            if any(word in command for word in ['hello', 'hi', 'hey']):
                greetings = [
                    "👋 Hello! How can I help you today?",
                    "👋 Hi there! What can I do for you?",
                    "👋 Hey! Ready to assist you.",
                    "👋 Hello! What would you like to know?"
                ]
                return random.choice(greetings)
            
            elif 'how are you' in command:
                responses = [
                    "🤖 I'm functioning well, thank you! How are you?",
                    "✨ All systems running smoothly! How can I help?",
                    "🎯 I'm doing great! What can I assist you with?"
                ]
                return random.choice(responses)
            
            elif any(phrase in command for phrase in ['who are you', 'what are you', 'your name']):
                return "🤖 I'm Aura, your AI voice assistant. I can help with tasks, answer questions, and make your day easier!"
            
            elif any(phrase in command for phrase in ['what can you do', 'help', 'capabilities']):
                return ("🚀 I can help you with weather, calculations, timers, web searches, file operations, "
                       "music control, translations, scheduling, navigation, and much more! "
                       "Just ask me naturally, like 'What's the weather?' or 'Set a timer for 10 minutes'.")
            
            elif any(word in command for word in ['goodbye', 'bye', 'exit', 'quit']):
                farewells = [
                    "👋 Goodbye! Have a wonderful day!",
                    "👋 See you later! Take care!",
                    "👋 Bye! Feel free to come back anytime.",
                    "👋 Farewell! Hope I was helpful!"
                ]
                response = random.choice(farewells)
                self.root.after(3000, self.on_closing)
                return response
            
            elif 'thank' in command:
                return "😊 You're very welcome! Happy to help anytime."
            
            else:
                unclear_responses = [
                    "🤔 I'm not sure I understand. Could you rephrase that?",
                    "❓ Could you clarify what you'd like me to do?",
                    "🎯 I didn't quite catch that. Please try again.",
                    "💭 Can you be more specific about what you need?",
                    "🤝 I'm here to help! What would you like to know or do?"
                ]
                return random.choice(unclear_responses)
                
        except Exception as e:
            logging.error(f"Conversation handling error: {e}")
            return "🤖 I'm here to help! What can I do for you?"

    def set_status(self, message):
        """Update status display"""
        try:
            self.status.config(text=message)
            logging.info(f"Status: {message}")
        except Exception as e:
            logging.error(f"Status update error: {e}")

    def add_to_history(self, text, tag=None):
        """Add text to conversation history"""
        try:
            self.history.config(state=tk.NORMAL)
            timestamp = datetime.datetime.now().strftime("%H:%M")
            
            self.history.insert(tk.END, f"[{timestamp}] ", "timestamp")
            
            if tag:
                self.history.insert(tk.END, f"{text}\n", tag)
            else:
                self.history.insert(tk.END, f"{text}\n")
            
            self.history.see(tk.END)
            self.history.config(state=tk.DISABLED)
            
            try:
                with open("conversation_log.txt", "a", encoding="utf-8") as f:
                    f.write(f"{datetime.datetime.now()}: {text}\n")
            except Exception:
                pass
                
        except Exception as e:
            logging.error(f"History update error: {e}")

    def _start_services(self):
        """Start background services"""
        greeting = self.get_greeting()
        self.set_status("🎯 System ready. Listening...")
        self.add_to_history(f"Aura: 🤖 {greeting}", "aura")
        threading.Thread(target=speak, args=(greeting,), daemon=True).start()

        self.play_sound()

        threading.Thread(target=self.voice_listener, daemon=True).start()
        threading.Thread(target=self.check_reminders, daemon=True).start()
        
        self.animate_orb()

    def on_closing(self):
        """Clean shutdown"""
        try:
            logging.info("Shutting down Aura Assistant...")
            self.animate = False
            
            self.set_status("👋 Goodbye! Thanks for using Aura")
            
            try:
                pygame.mixer.quit()
            except:
                pass
            
            try:
                if hasattr(tts_engine, 'engine') and tts_engine.engine:
                    tts_engine.engine.stop()
            except:
                pass
            
            self.root.quit()
            self.root.destroy()
            
        except Exception as e:
            logging.error(f"Shutdown error: {e}")
        
        finally:
            sys.exit(0)

# Unit Tests
class TestAuraAssistant(unittest.TestCase):
    def setUp(self):
        """Set up test environment"""
        self.root = tk.Tk()
        self.root.withdraw()
        self.app = AuraAssistant(self.root)

    def tearDown(self):
        """Clean up after tests"""
        self.app.animate = False
        self.root.destroy()

    def test_sanitize_filename(self):
        """Test filename sanitization"""
        self.assertEqual(sanitize_filename("test.txt"), "test.txt")
        self.assertEqual(sanitize_filename("test/../malicious.txt"), "testmalicious.txt")
        self.assertEqual(sanitize_filename("test<file>.txt"), "testfile.txt")
        self.assertEqual(sanitize_filename(""), "")

    def test_safe_eval(self):
        """Test safe mathematical evaluation"""
        self.assertEqual(safe_eval("2 + 2"), 4)
        self.assertEqual(safe_eval("sqrt(16)"), 4.0)
        with self.assertRaises(ValueError):
            safe_eval("import os")

    def test_timer_setting(self):
        """Test timer functionality"""
        result = self.app.set_timer("5")
        self.assertIn("Timer set for 5 minutes", result)
        
        result = self.app.set_timer("invalid")
        self.assertIn("specify duration", result)

    def test_command_processing(self):
        """Test basic command processing"""
        response = self.app.handle_conversation("hello")
        self.assertIn("Hello", response)
        
        response = self.app.handle_system_commands("what time is it")
        self.assertIn("current time", response)
        
        response = self.app.handle_calculation("calculate 2 plus 2")
        self.assertIn("4", response)

    def test_translation_feature(self):
        """Test translation functionality"""
        if TRANSLATION_AVAILABLE:
            response = self.app.handle_translation("translate hello to spanish")
            self.assertIn("hola" or "Hello", response.lower())

if __name__ == "__main__":
    try:
        if len(sys.argv) > 1 and sys.argv[1] == "test":
            unittest.main(argv=[sys.argv[0]], verbosity=2)
        else:
            root = tk.Tk()
            app = AuraAssistant(root)
            
            try:
                root.mainloop()
            except KeyboardInterrupt:
                logging.info("Application interrupted by user")
                app.on_closing()
            except Exception as e:
                logging.error(f"Application error: {e}")
                messagebox.showerror("Error", f"Application error: {str(e)}")
                app.on_closing()
                
    except Exception as e:
        logging.error(f"Startup error: {e}")
        print(f"Failed to start application: {e}")
        sys.exit(1)

SystemExit: 0